# Model2_TransferLearning.ipynb — Group 15 | 7PAM2033
# Model   : EfficientNetB4 (Transfer Learning)
# Dataset : MRI_Augmented (balanced — 4 classes)
# Purpose : Fine-tune EfficientNetB4 pre-trained on ImageNet for Alzheimer's
#           detection. Compare results against Model 1 (Baseline CNN).
#           EfficientNetB4 is chosen over ResNet50 because it achieves higher
#           accuracy with fewer parameters, making it better suited for
#           medical imaging tasks.

In [ ]:
# ================================================================================
# CELL 1 — Import libraries
# ================================================================================

import os
import random
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, accuracy_score)

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# Fix all seeds so results are reproducible
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13})

logger.info("TensorFlow version: %s", tf.__version__)


In [ ]:

# ================================================================================
# CELL 2 — Paths and hyperparameters
# ================================================================================

DATA_DIR    = Path("../Data/MRI_Augmented")
RESULTS_DIR = Path("../results")
MODELS_DIR  = Path("../results/models")
PLOTS_DIR   = Path("../results/plots")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

CLASSES    = ["NonDemented", "VeryMildDemented", "MildDemented", "ModerateDemented"]
N_CLASSES  = len(CLASSES)

# EfficientNetB4 expects 380x380 input by default but 224x224 works fine
# and is faster to train — we keep it consistent with Model 1
IMG_HEIGHT  = 224
IMG_WIDTH   = 224
IMG_SIZE    = (IMG_HEIGHT, IMG_WIDTH)
IMG_SHAPE   = (IMG_HEIGHT, IMG_WIDTH, 3)

BATCH_SIZE  = 32
EPOCHS_FROZEN  = 20    # Phase 1 — base layers frozen, only head trains
EPOCHS_FINETUNE = 30   # Phase 2 — top layers unfrozen for fine-tuning
VAL_SPLIT   = 0.15
LEARN_RATE_HEAD    = 0.001   # higher rate for new head layers
LEARN_RATE_FINETUNE = 0.0001  # lower rate for fine-tuning pretrained layers

logger.info("DATA_DIR = %s", DATA_DIR.resolve())

In [ ]:
# ================================================================================
# CELL 3 — Validate dataset and count images
# ================================================================================

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data folder not found: {DATA_DIR.resolve()}\n"
        "Please update DATA_DIR to your MRI_Augmented folder."
    )

print("Images per class:")
print("-" * 40)
total = 0
for cls in CLASSES:
    n = len(list((DATA_DIR / cls).glob("*.jpg")) +
            list((DATA_DIR / cls).glob("*.png")))
    print(f"  {cls:<22}: {n:>5}")
    total += n
print(f"\n  Total: {total:,}")


In [ ]:
# ================================================================================
# CELL 4 — Data generators
# ================================================================================
# EfficientNetB4 was pre-trained on ImageNet so images should be preprocessed
# the same way ImageNet images were — using EfficientNet's own preprocess_input.
# This rescales pixels from 0-255 to -1 to 1 which the model expects.

from tensorflow.keras.applications.efficientnet import preprocess_input

def efficientnet_preprocess(img):
    # Apply EfficientNet's own preprocessing instead of simple rescaling
    return preprocess_input(img)

train_datagen = ImageDataGenerator(
    preprocessing_function=efficientnet_preprocess,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True,
    zoom_range=0.05,
    validation_split=VAL_SPLIT
)

eval_datagen = ImageDataGenerator(
    preprocessing_function=efficientnet_preprocess
)

train_gen = train_datagen.flow_from_directory(
    str(DATA_DIR),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASSES,
    subset="training",
    shuffle=True,
    seed=SEED
)

val_gen = train_datagen.flow_from_directory(
    str(DATA_DIR),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASSES,
    subset="validation",
    shuffle=False,
    seed=SEED
)

test_gen = eval_datagen.flow_from_directory(
    str(DATA_DIR),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASSES,
    shuffle=False
)

print(f"\n  Train  : {train_gen.samples:,}")
print(f"  Val    : {val_gen.samples:,}")
print(f"  Test   : {test_gen.samples:,}")

In [ ]:
# ================================================================================
# CELL 5 — Build EfficientNetB4 model (Phase 1 — frozen base)
# ================================================================================
# Transfer learning works in two phases:
#   Phase 1: Freeze all EfficientNetB4 layers, only train our new head
#            The pre-trained features are preserved and we learn task-specific
#            patterns on top of them.
#   Phase 2: Unfreeze the top layers of EfficientNetB4 and fine-tune them
#            with a very low learning rate to adapt ImageNet features to MRI scans.

# Load EfficientNetB4 with ImageNet weights — exclude the top classification layer
# since we add our own for 4-class Alzheimer's detection
base_model = EfficientNetB4(
    weights="imagenet",       # use pre-trained ImageNet weights
    include_top=False,        # exclude the original 1000-class head
    input_shape=IMG_SHAPE
)

# Freeze all base model layers — we don't update these weights in Phase 1
base_model.trainable = False

print(f"EfficientNetB4 loaded.")
print(f"  Total layers  : {len(base_model.layers)}")
print(f"  Trainable     : {base_model.trainable}")

# Build the full model with our custom classification head
inputs  = layers.Input(shape=IMG_SHAPE)
x       = base_model(inputs, training=False)  # training=False keeps BatchNorm frozen
x       = layers.GlobalAveragePooling2D()(x)  # convert feature maps to 1D vector
x       = layers.BatchNormalization()(x)
x       = layers.Dense(256, activation="relu")(x)
x       = layers.Dropout(0.4)(x)              # dropout to prevent overfitting
outputs = layers.Dense(N_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARN_RATE_HEAD),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print(f"\nPhase 1 model built.")
print(f"  Trainable params  : {model.count_params():,}")

In [ ]:
# ================================================================================
# CELL 6 — Phase 1 Training — train only the head
# ================================================================================

phase1_callbacks = [
    callbacks.EarlyStopping(monitor="val_loss", patience=7,
                            restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(
        str(MODELS_DIR / "model2_efficientnet_phase1.h5"),
        monitor="val_accuracy", save_best_only=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                patience=3, min_lr=1e-7, verbose=1)
]

print("Phase 1: Training classification head (base frozen)...")
print("=" * 50)

history_phase1 = model.fit(
    train_gen,
    epochs=EPOCHS_FROZEN,
    validation_data=val_gen,
    callbacks=phase1_callbacks,
    verbose=1
)

print(f"\nPhase 1 complete.")
print(f"  Best val accuracy: {max(history_phase1.history['val_accuracy']):.4f}")

In [ ]:
# ================================================================================
# CELL 7 — Phase 2 Training — unfreeze top layers and fine-tune
# ================================================================================
# Now we unfreeze the top 30 layers of EfficientNetB4 and retrain with a
# very low learning rate. This adapts the high-level ImageNet features to
# MRI brain scan characteristics while preserving the lower-level features
# (edges, textures) which are universal across image types.

# Unfreeze the base model
base_model.trainable = True

# Only fine-tune the top 30 layers — lower layers learn universal features
# that don't need to change for MRI scans
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with a much lower learning rate to avoid destroying pre-trained weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARN_RATE_FINETUNE),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

unfrozen = sum(1 for l in model.layers if l.trainable)
print(f"Phase 2: Fine-tuning top 30 layers.")
print(f"  Trainable layers now: {unfrozen}")

phase2_callbacks = [
    callbacks.EarlyStopping(monitor="val_loss", patience=10,
                            restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(
        str(MODELS_DIR / "model2_efficientnet_best.h5"),
        monitor="val_accuracy", save_best_only=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                patience=5, min_lr=1e-8, verbose=1)
]

print("Phase 2: Fine-tuning...")
print("=" * 50)

history_phase2 = model.fit(
    train_gen,
    epochs=EPOCHS_FINETUNE,
    validation_data=val_gen,
    callbacks=phase2_callbacks,
    verbose=1
)

print(f"\nPhase 2 complete.")
print(f"  Best val accuracy: {max(history_phase2.history['val_accuracy']):.4f}")

In [ ]:
# ================================================================================
# CELL 8 — Plot training history for both phases
# ================================================================================
# Combine both phase histories into one continuous plot so we can see
# the full training story from frozen head to fine-tuning

all_acc     = history_phase1.history["accuracy"]     + history_phase2.history["accuracy"]
all_val_acc = history_phase1.history["val_accuracy"] + history_phase2.history["val_accuracy"]
all_loss    = history_phase1.history["loss"]         + history_phase2.history["loss"]
all_val_loss= history_phase1.history["val_loss"]     + history_phase2.history["val_loss"]
phase1_end  = len(history_phase1.history["accuracy"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(all_acc,     label="Train", color="#2196F3", linewidth=2)
axes[0].plot(all_val_acc, label="Val",   color="#FF9800", linewidth=2, linestyle="--")
axes[0].axvline(phase1_end, color="grey", linestyle=":", linewidth=1.5,
                label="Fine-tuning starts")
axes[0].set_title("Accuracy", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].set_ylim(0, 1)

# Loss
axes[1].plot(all_loss,     label="Train", color="#4CAF50", linewidth=2)
axes[1].plot(all_val_loss, label="Val",   color="#F44336", linewidth=2, linestyle="--")
axes[1].axvline(phase1_end, color="grey", linestyle=":", linewidth=1.5,
                label="Fine-tuning starts")
axes[1].set_title("Loss", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.suptitle("EfficientNetB4 — Training History (Phase 1 + Phase 2)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "model2_training_history.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ================================================================================
# CELL 9 — Evaluate on test set
# ================================================================================

print("Evaluating on test set...")
test_gen.reset()
y_pred_probs  = model.predict(test_gen, verbose=1)
y_pred        = np.argmax(y_pred_probs, axis=1)
y_true        = test_gen.classes

accuracy = accuracy_score(y_true, y_pred)
f1       = f1_score(y_true, y_pred, average="weighted")
y_true_oh = tf.keras.utils.to_categorical(y_true, N_CLASSES)
roc_auc   = roc_auc_score(y_true_oh, y_pred_probs,
                           multi_class="ovr", average="weighted")

print(f"\n  Accuracy  : {accuracy:.4f}  (KPI: ≥ 0.85)  {'✅' if accuracy >= 0.85 else '❌'}")
print(f"  F1 Score  : {f1:.4f}  (KPI: ≥ 0.85)  {'✅' if f1 >= 0.85 else '❌'}")
print(f"  ROC-AUC   : {roc_auc:.4f}  (KPI: ≥ 0.90)  {'✅' if roc_auc >= 0.90 else '❌'}")

print("\n  Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASSES))

In [ ]:
# ================================================================================
# CELL 10 — Confusion matrix
# ================================================================================

cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title("Confusion Matrix (Counts)", fontweight="bold")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].tick_params(axis="x", rotation=15)

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title("Confusion Matrix (Normalised)", fontweight="bold")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")
axes[1].tick_params(axis="x", rotation=15)

plt.suptitle("EfficientNetB4 — Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "model2_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ================================================================================
# CELL 11 — Save predictions and model
# ================================================================================

# Save predictions for fairness evaluation
pred_df = pd.DataFrame(y_pred_probs, columns=[f"prob_{c}" for c in CLASSES])
pred_df["predicted_class"] = y_pred
pred_df["predicted_label"] = [CLASSES[i] for i in y_pred]
pred_df["true_class"]      = y_true
pred_df["true_label"]      = [CLASSES[i] for i in y_true]
pred_df["correct"]         = (y_pred == y_true)
pred_df.to_csv(RESULTS_DIR / "model2_predictions.csv", index=False)
print("Predictions saved: results/model2_predictions.csv")

# Save final model
model.save(str(MODELS_DIR / "model2_efficientnet_final.h5"))
print("Model saved: results/models/model2_efficientnet_final.h5")

In [ ]:
# ================================================================================
# CELL 12 — Compare Model 1 vs Model 2
# ================================================================================
# Load Model 1 predictions and compare side by side

m1_path = RESULTS_DIR / "model1_predictions.csv"
if m1_path.exists():
    m1_df = pd.read_csv(m1_path)
    m1_acc = accuracy_score(m1_df["true_class"], m1_df["predicted_class"])
    m1_f1  = f1_score(m1_df["true_class"], m1_df["predicted_class"], average="weighted")

    print("\n" + "=" * 55)
    print("  MODEL COMPARISON — Model 1 vs Model 2")
    print("=" * 55)
    print(f"  {'Metric':<12} {'Model 1 CNN':>14} {'Model 2 EffNet':>16}")
    print(f"  {'-'*12} {'-'*14} {'-'*16}")
    print(f"  {'Accuracy':<12} {m1_acc:>14.4f} {accuracy:>16.4f}")
    print(f"  {'F1 Score':<12} {m1_f1:>14.4f} {f1:>16.4f}")
    print(f"  {'ROC-AUC':<12} {'N/A':>14} {roc_auc:>16.4f}")
    improvement = (accuracy - m1_acc) / m1_acc * 100
    print(f"\n  EfficientNetB4 accuracy improvement: {improvement:+.2f}%")
else:
    print("Model 1 predictions not found — run Model1_Baseline_CNN.py first.")


In [ ]:
# ================================================================================
# CELL 13 — Summary
# ================================================================================

print("\n" + "=" * 60)
print("  MODEL 2 — EFFICIENTNETB4 SUMMARY")
print("=" * 60)
print(f"""
  ARCHITECTURE
  ------------
  Base model  : EfficientNetB4 (ImageNet weights)
  Fine-tuned  : Top 30 layers
  Input size  : {IMG_HEIGHT}x{IMG_WIDTH}x3
  Parameters  : {model.count_params():,}

  TRAINING
  --------
  Phase 1 epochs : {len(history_phase1.history['loss'])} (head only)
  Phase 2 epochs : {len(history_phase2.history['loss'])} (fine-tuning)
  Best val acc   : {max(all_val_acc):.4f}

  TEST RESULTS
  ------------
  Accuracy  : {accuracy:.4f}  {'✅' if accuracy >= 0.85 else '❌'}
  F1 Score  : {f1:.4f}  {'✅' if f1 >= 0.85 else '❌'}
  ROC-AUC   : {roc_auc:.4f}  {'✅' if roc_auc >= 0.90 else '❌'}

  SAVED FILES
  -----------
  Model     : results/models/model2_efficientnet_final.h5
  Preds     : results/model2_predictions.csv
  Plots     : results/plots/model2_training_history.png
              results/plots/model2_confusion_matrix.png

  NEXT STEP : Model3_Hybrid_Multimodal.ipynb
""")
